# European Dynamics in EU public procurement - collection, checks and model
One notebook for the parts that are not in SQL or Power BI: what was collected, the data checks, a few answers straight from the warehouse, and the single-offer model. Run the pipeline first (README, *How to rerun*).

In [1]:
import sys, json, warnings
from pathlib import Path
import pandas as pd
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from load_sql import connect, DATABASE
cn = connect(DATABASE)
ST = ROOT / 'data' / 'processed' / 'staging'
pd.set_option('display.max_colwidth', 80)

## 1. Collection (official sources only, low frequency)
* **TED legacy CSV** (data.europa.eu, *Contract award notices* 2016-2023): one row per award.
* **TED Search API v3** + notice **XML** (eForms, Nov 2023 - Nov 2026): lot-level results of the notices naming European Dynamics; notice-level CPV 72 context of the same buyers and of EU institutions.
* **FTS** yearly datasets 2016-2025 (European Commission).
* **ECB** annual euro reference rates; **CPV 2008** labels.
Scripts: `src/download_ted.py`, `src/download_fts.py`, filtering in `src/build_dataset.py`.

In [2]:
sizes = {f.name: len(pd.read_csv(f, dtype=str)) for f in sorted(ST.glob('*.csv'))}
pd.Series(sizes, name='rows').to_frame()

,rows
ecb_exr_annual.csv,118
fts_ed.csv,5549
ref_country.csv,42
ref_cpv.csv,148
sim_agency.csv,6
sim_service_request.csv,48000
sim_user_agency.csv,8
ted_eforms_ed_awards.csv,125
ted_eforms_market_notices.csv,1082
ted_legacy_awards.csv,5461


## 2. Data checks
The warehouse runs 16 checks (`sql/06_quality_checks.sql`). PASS / WARN / FAIL:

In [3]:
pd.read_sql('SELECT CheckId, CheckName, Actual, Status FROM qa.CheckResult ORDER BY CheckId', cn)

,CheckId,CheckName,Actual,Status
0,QA01,Years 2016-2025 without any European Dynamics award,0.00,PASS
1,QA02,Legacy award rows removed as exact duplicates (staging - fact),42.00,PASS
2,QA03,FTS contracted EUR: fact minus staging,0.00,PASS
3,QA04,Award rows without a CPV key,0.00,PASS
4,QA05,Award rows without a buyer country,0.00,PASS
5,QA06,Award rows with an unknown buyer,0.00,PASS
6,QA07,Notice-lots with more than one counted value,0.00,PASS
7,QA08,Award values removed as outliers (> EUR 2 bn or < EUR 1),325.00,PASS
8,QA09,% of European Dynamics award rows without any value,9.57,PASS
9,QA10,eForms rows excluded (winner reference = buyer's service provider),70.00,PASS


### Name resolution
Raw spellings of European Dynamics entities and the entity they resolve to (alias table `ref.EntityAlias`, LIKE patterns on a normalised name).

In [4]:
pd.read_sql('''SELECT TOP 12 RawName, EntityName, SourceSystem, Occurrences FROM pbi.vEntityNameMap
                ORDER BY Occurrences DESC''', cn)

,RawName,EntityName,SourceSystem,Occurrences
0,EUROPEAN DYNAMICS LUXEMBOURG SA*ED LUXEMBOURG,European Dynamics Luxembourg SA,FTS,3937
1,EUROPEAN DYNAMICS LUXEMBOURG SA,European Dynamics Luxembourg SA,FTS,683
2,EUROPEAN DYNAMICS ADVANCED INFORMATION TECHNOLOGY AND TELECOMMUNICATION SYST...,European Dynamics SA (Greece),FTS,507
3,CONSORTIUM EUROPEAN DYNAMICS*EUROPEAN DYNAMICS LUXEMBOURG SA EUROPEANDYNAMIC...,European Dynamics Luxembourg SA,FTS,360
4,European Dynamics Luxembourg SA,European Dynamics Luxembourg SA,TED legacy CSV,65
5,European Dynamics SA,European Dynamics SA (Greece),TED legacy CSV,54
6,EUROPEAN DYNAMICS ADVANCED SYSTEMSOF TELECOMMUNICATIONS INFORMATICS AND TELE...,European Dynamics SA (Greece),FTS,36
7,EUROPEAN DYNAMICS BELGIUM,European Dynamics Belgium SA,FTS,25
8,EUROPEAN DYNAMICS LUXEMBOURG SA,European Dynamics Luxembourg SA,TED eForms XML,24
9,European Dynamics Luxembourg S.A.,European Dynamics Luxembourg SA,TED legacy CSV,16


### eForms rows excluded (DQ-03 / DQ-04)
In some eForms notices the winner reference points to the organisation that the same notice lists as the buyer's **service provider** (a European Dynamics entity), while the tendering party is another company or the contract is not IT. These rows are excluded, not counted as awards.

In [5]:
x = pd.read_csv(ST / 'ted_eforms_ed_awards.csv', dtype=str)
x['rule'] = x.dq_excluded.fillna('kept').str[:5]
x.groupby(['rule', 'buyer_country']).size().unstack(fill_value=0)

buyer_country,BEL,CYP,DEU,DNK,EST,FRA,GRC,IRL,LTU,LUX,MLT,NLD,PRT,SWE
rule,,,,,,,,,,,,,,
DQ-03,0,0,0,0,0,0,0,53,10,0,0,0,0,0
DQ-04,0,0,0,0,0,0,0,5,2,0,0,0,0,0
kept,10,1,9,2,2,1,2,1,1,8,2,1,13,2


## 3. Answers from the warehouse (SQL, window functions)

In [6]:
pd.read_sql('''WITH y AS (SELECT d.[Year], COUNT(*) AS Awards FROM dwh.FactAward f JOIN dwh.DimDate d ON d.DateKey = f.DateKey
                         WHERE f.IsED = 1 GROUP BY d.[Year])
SELECT [Year], Awards, AVG(1.0 * Awards) OVER (ORDER BY [Year] ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS MovingAvg3Y
FROM y ORDER BY [Year]''', cn).round(1)

,Year,Awards,MovingAvg3Y
0,2016,26,26.0
1,2017,11,18.5
2,2018,18,18.3
3,2019,14,14.3
4,2020,22,18.0
5,2021,21,19.0
6,2022,20,21.0
7,2023,24,21.7
8,2024,11,18.3
9,2025,19,18.0


In [7]:
pd.read_sql('''SELECT p.ProcedureFamily, SUM(IIF(f.IsED = 1, 1, 0)) AS EdLots,
  AVG(IIF(f.IsED = 1, IIF(f.NumberOfOffers = 1, 1.0, 0.0), NULL)) AS EdSingleOffer,
  AVG(IIF(f.IsED = 0, IIF(f.NumberOfOffers = 1, 1.0, 0.0), NULL)) AS OtherSingleOffer
FROM dwh.FactAward f JOIN dwh.DimProcedure p ON p.ProcedureKey = f.ProcedureKey
WHERE f.NumberOfOffers IS NOT NULL GROUP BY p.ProcedureFamily ORDER BY EdLots DESC''', cn).round(3)

,ProcedureFamily,EdLots,EdSingleOffer,OtherSingleOffer
0,Open,140,0.043,0.127
1,Negotiated,26,0.577,0.806
2,Restricted,14,0.071,0.057
3,Competitive dialogue,2,0.000,0.000
4,Other,2,0.000,0.000


## 4. Model: will an IT-services lot receive a single offer?
Target: `NumberOfOffers = 1` on CPV 72 lots (all suppliers). Features known at publication: procedure, framework, buyer type and country, CPV group, log value, year, TED format. Missing values imputed, categories one-hot encoded (rare levels grouped), 5-fold stratified cross-validation, permutation importance on a held-out 25%.

In [8]:
import ml_single_offer
ml_single_offer.main()

{
 "rows": 4748,
 "single_offer_rate": 0.2005054759898905,
 "models": {
  "Logistic regression": {
   "roc_auc": 0.867436045391804,
   "roc_auc_sd": 0.008975896105096802,
   "avg_precision": 0.6944942783152425
  },
  "Gradient boosting": {
   "roc_auc": 0.8946663257875993,
   "roc_auc_sd": 0.010134893804463687,
   "avg_precision": 0.7562830361144697
  }
 },
 "best_model": "Gradient boosting",
 "ed_single_offer_rate": 0.10919540229885058,
 "ed_rows": 174
}
             feature  importance  importance_sd  rank
           Procedure    0.146038       0.008267     1
       Buyer country    0.085299       0.008998     2
Contract value (log)    0.034559       0.005120     3
          Buyer type    0.023851       0.006435     4
           CPV group    0.014328       0.003240     5
 Framework agreement    0.011463       0.003442     6
                Year    0.004722       0.004324     7
          TED format   -0.000216       0.000445     8


**Reading.** The procedure type carries most of the signal: negotiated procedures without a prior call are, by design, often single-offer. Buyer country comes next (national market structure), then value. **Limits:** notices are not independent (same buyers, same frameworks), the number of offers is missing for part of the notices, eForms market data are at notice level, and the model describes patterns - it does not judge any procurement or supplier. A single offer is not an anomaly in itself.